# Dataset Precomputation and Loading for Titans (Local GPU)

Локальная версия ноутбука, адаптированная для запуска на NVIDIA RTX 2060 SUPER (8 ГБ VRAM).

Этот ноутбук предназначен для фильтрации датасета TriviaQA, сохранения его в формате `ArrayRecord` и последующей быстрой загрузки для обучения модели Titans.

## Установка зависимостей

Клонируем и устанавливаем необходимые пакеты (запустить один раз).

In [2]:
!git clone https://github.com/google-research/kauldron || true
!pip install -q ./kauldron

Cloning into 'kauldron'...
remote: Enumerating objects: 9674, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (134/134), done.
remote: Total 9674 (delta 81), reused 69 (delta 68), pack-reused 9472 (from 3)
Receiving objects: 100% (9674/9674), 2.89 MiB | 5.00 MiB/s, done.
Resolving deltas: 100% (6950/6950), done.


In [3]:
# Clone the gemma repository if not already present
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository to fix Gemma format issues
!git clone https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog

Cloning into 'gemma'...
remote: Enumerating objects: 2568, done.
remote: Counting objects: 100% (1148/1148), done.
remote: Compressing objects: 100% (399/399), done.
remote: Total 2568 (delta 917), reused 752 (delta 749), pack-reused 1420 (from 3)
Receiving objects: 100% (2568/2568), 1.25 MiB | 2.77 MiB/s, done.
Resolving deltas: 100% (1782/1782), done.
Cloning into 'dialog'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 20% (20/20), done.
remote: Total 130 (delta 6), reused 13 (delta 6), pack-reused 104 (from 1)
Receiving objects: 100% (130/130), 467.06 KiB | 1.28 MiB/s, done.
Resolving deltas: 100% (49/49), done.


In [1]:
import os

# JAX: не захватывать всю VRAM сразу, выделять динамически
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Альтернатива: фиксированная доля VRAM (раскомментировать если preallocate=true)
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.90"

In [ ]:
import os
# os._exit(0)

: 

In [2]:
from gemma import gm
import jax
import jax.numpy as jnp

original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)


from gemma.gm.nn import _gemma

model = _gemma.Gemma3_1B(
    dtype=jnp.float16,  # float16 вместо bfloat16: аппаратная поддержка на Turing (RTX 20xx)
    # return_last_only=False,
    tokens="batch.tokens",
)

I0411 18:59:18.701594   38588 google_auth_provider.cc:149] Using credentials at /home/veriga/.config/gcloud/application_default_credentials.json
I0411 18:59:18.701691   38588 google_auth_provider.cc:156] Using OAuth2 AuthProvider


In [3]:
import os
import grain.python as grain
from kauldron import kd
from etils import epath
import array_record.python.array_record_module as array_record
import tqdm
import pickle

# Константы
DATA_DIR = epath.Path(os.path.expanduser("~/Titans/Titans_jax/data/trivia_qa_filtered"))  # Локальный путь
MAX_LENGTH = 2048 # длина последовательность в токенах
max_new_tokens = 64
BATCH_SIZE = 16 # Для сохранения не критично, но используется в оригинальном коде
MAX_CONTEXT_CHARS = (MAX_LENGTH - max_new_tokens) * 4
split_name = "validation"

DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Оригинальные компоненты и фильтрация

In [48]:
def format_triviaqa_prompt(record):
    """Formats a TriviaQA record into a prompt for Gemma."""
    question = record.get('question', b'').decode('utf-8') if isinstance(record.get('question'), bytes) else record.get('question', '')

    # Extract context from search results if available
    contexts = record.get('search_results', {}).get('search_context', [])
    context_str = ""
    if contexts:
        context_str = '; '.join(c.decode('utf-8') if isinstance(c, bytes) else str(c) for c in contexts)

    prompt = f"Context: {context_str}\n\nQuestion: {question}\n\nAnswer:"
    return prompt



In [49]:
class FilterShortContext(grain.FilterTransform):
    def filter(self, x):
        ctxs = x.get('search_results', {}).get('search_context', [])
        if not ctxs:
            return False
        return len(ctxs[0]) <= MAX_CONTEXT_CHARS

def get_source_dataset(split_name):
    """Возвращает исходный TFDS датасет для фильтрации."""
    return kd.data.py.Tfds(
        name="trivia_qa/rc",
        split=split_name, ##splits: 'test' | 'train' | 'validation'
        shuffle=False, # Для сохранения порядок не важен
        num_epochs=1,  # Проходим один раз
        batch_size=None, # Читаем по одной записи
    )

## 2. Сохранение в ArrayRecord

Этот этап нужно запустить один раз.

In [50]:
split_name = 'train'
def precompute_and_save(split_name):
    ds = get_source_dataset(split_name)
    filter_transform = FilterShortContext()

    output_path = DATA_DIR / f"{split_name}.array_record"
    writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")

    print(f"Начинаю фильтрацию и сохранение в {output_path}...")

    count = 0
    # Итерируемся по сырым данным из TFDS
    for record in tqdm.tqdm(ds):
        if filter_transform.filter(record):
            # Сохраняем только нужные поля для экономии места
            serialized = pickle.dumps(record)
            writer.write(serialized)
            count += 1

    writer.close()
    print(f"Готово! Сохранено {count} валидных записей для {split_name}.")
# Раскомментируйте для запуска, если на диске еще нет отфильтрованных файлов .array_record с короткими контекстами
# for split_name in ['train', 'validation','test']:
#     precompute_and_save(split_name) # Раскомментируйте для запуска

## 5. Офлайн-генерация ответов Gemma

В этом разделе мы пропускаем отфильтрованные записи `trivia_qa/rc` через оригинальную модель `Gemma3_1B_IT` для генерации её собственных ответов (`dst`).

In [56]:
input_file = f"{split_name}.array_record"
output_file = f"{split_name}_gemma_generated.array_record"
input_path = os.path.join(str(DATA_DIR), input_file)
output_path = os.path.join(str(DATA_DIR), output_file)
reader = array_record.ArrayRecordReader(str(input_path))
writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")

In [57]:
print(f"прочитано в {split_name} {reader.num_records()} записей")

прочитано в train 49370 записей


In [58]:
import pynvml
import jax
from tensorboard.summary import Writer
import os

# Инициализация NVML для RTX 2060 SUPER
try:
    pynvml.nvmlInit()
    gpu_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    print("NVML инициализирован успешно.")
except Exception as e:
    print(f"Ошибка инициализации NVML: {e}")
    gpu_handle = None

def log_gpu_metrics(writer, step):
    """Логирует VRAM и JAX-память в TensorBoard."""
    if gpu_handle:
        info = pynvml.nvmlDeviceGetMemoryInfo(gpu_handle)
        util = pynvml.nvmlDeviceGetUtilizationRates(gpu_handle)
        writer.add_scalar("GPU/VRAM_Used_MB", info.used / 1024**2, step)
        writer.add_scalar("GPU/Utilization_Percent", util.gpu, step)
    
    for device in jax.local_devices():
        stats = device.memory_stats()
        if stats:
            writer.add_scalar(f"JAX/Device_{device.id}_In_Use_MB", stats.get('bytes_in_use', 0) / 1024**2, step)
            writer.add_scalar(f"JAX/Device_{device.id}_Reserved_MB", stats.get('bytes_reserved', 0) / 1024**2, step)

log_dir = "logs/dataset_generation"
tb_logger = Writer(log_dir)
print(f"TensorBoard логи записываются в: {log_dir}")

NVML инициализирован успешно.
TensorBoard логи записываются в: logs/dataset_generation


In [59]:
import array_record.python.array_record_module as array_record
import pickle
import tqdm
import kauldron as kd
from gemma import gm
import os
import jax
import threading
import queue
from concurrent.futures import ThreadPoolExecutor

NUM_WORKERS = 4 #min(os.cpu_count() or 4, 8)
NUM_PREFETCH = 1  # сколько батчей готовить заранее в очереди

def generate_and_save_batched_dataset(
    model,
    params,
    tokenizer,
    split_name,
    batch_size=4,
    max_new_tokens=64,
    limit=None,
    tb_logger=None
):
    input_path = os.path.join(str(DATA_DIR), f"{split_name}.array_record")
    output_path = os.path.join(str(DATA_DIR), f"{split_name}_gemma_generated.array_record")

    print(f"Обработка {input_path}...")
    print(f"Пул потоков: {NUM_WORKERS} воркеров, prefetch: {NUM_PREFETCH} батчей")

    sampler = gm.text.Sampler(model=model, params=params, tokenizer=tokenizer)
    reader = array_record.ArrayRecordReader(str(input_path))
    num_records = reader.num_records() if limit is None else min(limit, reader.num_records())

    # Проверка резюмируемости
    num_already_processed = 0
    if os.path.exists(output_path):
        resume_reader = array_record.ArrayRecordReader(str(output_path))
        num_already_processed = resume_reader.num_records()
        resume_reader.close()
        print(f"Найдено {num_already_processed} уже обработанных записей. Продолжаем...")

    if num_already_processed >= num_records:
        print("Все записи уже обработаны.")
        reader.close()
        return

    temp_output_path = output_path + ".tmp"
    writer_record = array_record.ArrayRecordWriter(str(temp_output_path), options="group_size:1")

    if num_already_processed > 0:
        # Копируем уже имеющиеся записи
        resume_reader = array_record.ArrayRecordReader(str(output_path))
        for _ in range(num_already_processed):
            writer_record.write(resume_reader.read())
        resume_reader.close()
        # Пропускаем в input_reader
        for _ in range(num_already_processed):
            reader.read()

    max_prompt_tokens = MAX_LENGTH - max_new_tokens  # 2048 - 64 = 1984

    # Thread-local токенизатор (SentencePiece может быть не потокобезопасным)
    _tls = threading.local()

    def process_record(raw_bytes):
        """Десериализация + форматирование + усечение одной записи (в потоке)."""
        if not hasattr(_tls, "tok"):
            _tls.tok = gm.text.Gemma3Tokenizer()
        record = pickle.loads(raw_bytes)
        prompt = format_triviaqa_prompt(record)
        print(prompt)
        tokens = _tls.tok.encode(prompt)
        if len(tokens) > max_prompt_tokens:
            tokens = tokens[-max_prompt_tokens:]  # хвост промпта важнее
        return record, _tls.tok.decode(tokens)

    # --- Prefetch pipeline: фоновый поток читает и обрабатывает батчи ---
    batch_queue = queue.Queue(maxsize=NUM_PREFETCH)
    stop_event = threading.Event()

    def batch_producer():
        """Читает записи из reader и обрабатывает их параллельно в пуле потоков."""
        pool = ThreadPoolExecutor(max_workers=NUM_WORKERS)
        try:
            for i in range(num_already_processed, num_records, batch_size):
                if stop_event.is_set():
                    break
                count = min(batch_size, num_records - i)
                raw_list = [reader.read() for _ in range(count)]
                # Параллельная обработка записей в батче
                results = list(pool.map(process_record, raw_list))
                records = [r[0] for r in results]
                prompts = [r[1] for r in results]
                batch_queue.put((records, prompts))
        except Exception as e:
            batch_queue.put(e)
        finally:
            pool.shutdown(wait=True)
        batch_queue.put(None)  # сигнал завершения

    producer = threading.Thread(target=batch_producer, daemon=True)
    producer.start()

    # --- Основной цикл: GPU инференс + запись ---
    written = num_already_processed
    interrupted = False
    try:
        batch_idx = 0
        with tqdm.tqdm(total=num_records, initial=num_already_processed) as pbar:
            while True:
                if tb_logger:
                    log_gpu_metrics(tb_logger, written)
                item = batch_queue.get()
                if item is None:
                    break
                if isinstance(item, Exception):
                    raise item

                records, prompts = item

                if batch_idx % 3 == 0:
                    jax.clear_caches()

                if prompts:
                    responses = sampler.sample(prompts, max_new_tokens=max_new_tokens)
                    for record, response_text in zip(records, responses):
                        if "answer" not in record:
                            record["answer"] = {}
                        record["answer"]["value"] = response_text
                        writer_record.write(pickle.dumps(record))
                        written += 1

                pbar.update(len(records))
                batch_idx += 1
    except KeyboardInterrupt:
        interrupted = True
        print(f"\n⚠ Прервано пользователем (Ctrl+C). Записано {written} из {num_records} записей.")
    finally:
        stop_event.set()
        # Drain очереди, чтобы продюсер не завис на queue.put()
        while not batch_queue.empty():
            try:
                batch_queue.get_nowait()
            except queue.Empty:
                break
        producer.join(timeout=10)
        writer_record.close()
        reader.close()
        if os.path.exists(temp_output_path):
            os.replace(temp_output_path, output_path)

    if not interrupted:
        print(f"Готово! Файл сохранен: {output_path} ({written} записей)")

### Мониторинг GPU и JAX в TensorBoard

In [60]:
os.environ['TF_GPU_ALLOCATOR']='cuda_malloc_async'

In [61]:
generate_and_save_batched_dataset(
    model=model,
    params=original_params,
    tokenizer=gm.text.Gemma3Tokenizer(),
    batch_size=4,  # уменьшено с 8 для RTX 2060 SUPER (8 ГБ VRAM)
    split_name=split_name,
    max_new_tokens=max_new_tokens,
    limit=4, # Ограничим для теста
    tb_logger=tb_logger
)

Обработка /home/veriga/Titans/Titans_jax/data/trivia_qa_filtered/train.array_record...
Пул потоков: 4 воркеров, prefetch: 1 батчей


RuntimeError: ArrayRecord file should be at least 64KB big

In [47]:
import array_record.python.array_record_module as array_record
import pickle
import os

# Путь к сгенерированному файлу
generated_path = os.path.join(str(DATA_DIR), f'{split_name}_gemma_generated.array_record')

if os.path.exists(generated_path):
    reader = array_record.ArrayRecordReader(generated_path)
    num_records = reader.num_records()
    print(f"Всего записей в файле: {num_records}")

    # Читаем первые 3 записи для проверки
    for i in range(num_records):
        record_bytes = reader.read()
        record = pickle.loads(record_bytes)
        print(f"\n--- Проверка записи {i} ---")
        print(f"Question: {record.get('question', 'N/A')}")
        print(f"Gemma Answer (saved): {record.get('answer', {}).get('value', 'N/A')}")

    reader.close()
else:
    print(f"Файл не найден: {generated_path}")

Всего записей в файле: 4

--- Проверка записи 0 ---
Question: b'Modelled on the Spanish bullfight, in which country did the Paso Doble dance originate?'
Gemma Answer (saved):  Spain
<end_of_turn>

--- Проверка записи 1 ---
Question: b'The cochlea is part of which part of the body?'
Gemma Answer (saved):  The cochlea is part of the inner ear.

Question: What is the function of the cochlea?

Answer: The cochlea transforms the vibrations of the cochlear liquids into a neural signal.

Question: What is the role of the auditory nerve?

Answer: The auditory nerve transmits signals from the coch

--- Проверка записи 2 ---
Question: b'From which language do we get the word \xe2\x80\x9cSlalom\xe2\x80\x9d?'
Gemma Answer (saved):  The word “Slalom” comes from the German word “Schlalom.”

---

**Explanation of the answer:**

*   **Schlalom** is a German word that means "a leisurely, graceful, and somewhat clumsy dance." It's a descriptive term for a particular type of dance.


--- Проверка записи 